# Task 1.1  Pauli Operators

## 목표:  Pauli Operators 이해하기

**개요:** 이 노트북은 Pauli operator가 무엇이고, 어떻게 initialize하고, 어떤 attributes와 methods가 있는지 등을 다룹니다.

Pauli operator는 quantum computing의 기초구성요소로서, single/multi-qubit operations을 표현합니다. (Pauli operator가 single qubit에 어떻게 작용하는지 visualize하려면 Bloch sphere (https://javafxpert.github.io/grok-bloch/)가 도움이 됩니다.)

Qiskit에서는 `Pauli` class (https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Pauli)가 Pauli operator를 다루는 다양한 방법을 제공합니다.

In [1]:
# 필요한 모듈들을 import
from qiskit.quantum_info import Pauli, ScalarOp
import numpy as np
from qiskit import QuantumCircuit

print("Libraries imported successfully.")

Libraries imported successfully.


### Initialize 하기

Qiskit `Pauli` class는 다음과 같은 방식으로 instantiate할 수 있습니다.

#### 1. Pauli String 방식

Pauli matrix를 나타내는 문자열로 Pauli operator를 만들수 있습니다:
- `I`: Identity operator
- `X`: Pauli-X (bit-flip) operator
- `Y`: Pauli-Y operator
- `Z`: Pauli-Z (phase-flip) operator

Note: Qiskit은 qubit을 문자열로 나타낼때 qubit (n-1)이 가장 왼쪽에 오고 qubit 0가 가장 오른쪽에 오는 <b>Little Endian 표기법</b>을 사용합니다.

Phase는 ` ` (default), `-`, `i`, `-i`로 나타냅니다.

In [2]:
# 문자열로 Pauli operator 만들기
p = Pauli('XZ')      # default phase (+1)
p2 = Pauli('iYY')    # imaginary phase i
p3 = Pauli('-iZX')   # negative imaginary phase -i

print(f"P:{p}, P2:{p2}, P3:{p3}")

P:XZ, P2:iYY, P3:-iZX


#### 2. Boolean Array 방식

Boolean array로 Pauli operator를 만들수 있습니다: 
- `z` array는 각각의 qubit에 Pauli Z가 apply 되는지 나타냅니다. (True = Z operator, False = I operator)
- `x` array는 각각의 qubit에 Pauli X가 apply 되는지 나타냅니다 (True = X operator, False = I operator)

각각의 qubit에 실제 작용하는 Pauli operator는 다음과 같이 Pauli algebra에 의해 결정됩니다:
- (z=False, x=False) → I (Identity)
- (z=False, x=True) → X
- (z=True, x=False) → Z
- (z=True, x=True) → Y (since Y = iXZ)

In [3]:
# qubit 0: (z=False, x=False) → I
# qubit 1: (z=False, x=True) → X 
# qubit 2: (z=True, x=False) → Z
# qubit 3: (z=True, x=True) → Y
z = np.array([False,False,True,True])  
x = np.array([False,True,False,True])  

pauli_array_rep = Pauli((z, x))

# Little Endian 표기법때문에 `print` 결과는 문자열의 순서가 뒤집힙니다.
print(f"Pauli Operator from Boolean Array: {pauli_array_rep}")

Pauli Operator from Boolean Array: YZXI


#### 3. Quantum Circuit 방식

Pauli gate (I, X, Y, Z) 만으로 만들어진 quantum circuit으로도 Pauli operator를 만들수 있습니다.

In [4]:
# Pauli gate로 quantum circuit을 만듭니다.
qc = QuantumCircuit(4)
qc.id(0) # I gate on qubit 0
qc.z(1)  # Z gate on qubit 1
qc.x(2)  # X gate on qubit 2
qc.y(3)  # Y gate on qubit 3
# Quantum circuit으로 Pauli operator를 만들려면 다른 종류의 gate를 사용할수 없습니다. 다음 line을 uncomment하면 error 납니다.
#qc.cx(0,1)

# Quantum circuit으로 Pauli operator를 만듭니다.
pauli_quantum_circuit = Pauli(qc)

print(f"Pauli Operator from quantum circuit : {pauli_quantum_circuit}")

Pauli Operator from quantum circuit : YXZI


#### 4. ScalarOp 방식

Pauli operator는 scalar operation과 합쳐질수 있습니다. Qiskit `ScalarOp` class는 identity operator에 scalar가 곱해진 것인데, global phase를 적용하기위해 Pauli operator와 합쳐질수 있습니다.

In [5]:
# Scalar operator (scaled identity)를 만듭니다.
# Pauli의 phase는 1, -1j, -1, 1j만 가능하므로, `coeff`를 이 넷 중에서 고릅니다. 
scalar_op = ScalarOp(dims=(2,2,2,2), coeff=-1j)  # -1*j

# Pauli operator를 만듭니다.
pauli_scalarop = Pauli('YXZI')

# 둘을 "compose" 하면 Pauli operator에 global phase가 붙습니다.
p = scalar_op.compose(pauli_scalarop)

print(f"Pauli Operator Before applying Scalar : {pauli_scalarop}")

print(f"Pauli Operator from ScalarOp : {p}")

Pauli Operator Before applying Scalar : YXZI
Pauli Operator from ScalarOp : -iYXZI


### 표현방식

#### 1. Matrix로 나타내기

`to_matrix()` method는 Pauli operator를 matrix 형태로 바꿔줍니다. 하나 이상의 qubit에 작용하는 Pauli operator의 경우, 각각의 Pauli matirx의 tensor product로 나타내집니다.

In [6]:
# Pauli operator를 matrix로 바꾸기
p = Pauli('ZX')
p_matrix = p.to_matrix()

print(f"Matrix Format: {p_matrix}")

Matrix Format: [[ 0.+0.j  1.+0.j  0.+0.j  0.+0.j]
 [ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j -1.+0.j]
 [ 0.+0.j  0.+0.j -1.+0.j  0.+0.j]]


#### 2. 문자열로 나타내기

`to_label()` method는 Pauli operator를 (I, X, Y, Z)로 구성된 문자열로 바꿔줍니다.

In [7]:
# 문자열로 바꾸기
print(f"String format:{str(p)}")

print(f"Label format:{p.to_label()}")

String format:ZX
Label format:ZX


### `Pauli` class 속성들

`Pauli` class는 Pauli operator의 여러 성질을 사용할수 있도록 다음과 같은 속성들을 제공합니다:
- `dim`: operator의 dimension 
- `num_qubits`: qubit의 갯수
- `num_clbits`: classical bit의 갯수 (Pauli operator의 경우 항상 0.)
- `phase`: operator의 phase (0, 1, 2, 3은 각각 +1, i, -1, -i을 나타냅니다.)
- `x`: X boolean array
- `z`: Z boolean array

In [8]:
# `Pauli` class 속성 사용하기
print(f"Dimension: {p.dim}")  
print(f"Classical bits: {p.num_clbits}, Qubits: {p.num_qubits}")  
print(f"X: {p.x}, Z: {p.z}, Phase: {p.phase}")  

Dimension: (4, 4)
Classical bits: 0, Qubits: 2
X: [ True False], Z: [False  True], Phase: 0


### `Pauli` class 함수들

`Pauli` class는 Pauli operator를 다루기 위한 다양한 함수들을 제공합니다.

#### adjoint

주어진 Pauli operator의 adjoint (conjugate transpose)를 만듭니다. Pauli operator는 Hermitian이므로 adjoint는 자기 자신과 같지만, global phase는 바뀔수 있습니다.

In [9]:
# Pauli operator의 adjoint 만들기
p.adjoint()
print(f"pauli operator:{p3}, adjoint: {p3.adjoint()}")

pauli operator:-iZX, adjoint: iZX


#### anticommutes

주어진 두 Pauli operator A, B가 anticommute한지, 즉 AB = -BA인지 확인합니다.

In [10]:
# 두 Pauli operator가 anticommute한지 확인하기
p = Pauli('X')      # X
p2 = Pauli('Y')     # Y
print(f"{p} anticommutes with {p2} is {p.anticommutes(p2)}")

X anticommutes with Y is True


#### compose

주어진 두 Pauli operator의 composition (left matrix product)을 구합니다. 결과는 phase x 다른 Pauli operator입니다.
(A.compose(B) == A & B == BA)

In [11]:
# 두 Pauli operator의 composition 구하기

print(f"{p} compose with {p2} is {p.compose(p2)}")

X compose with Y is -iZ


#### conjugate

주어진 Pauli operator의 complex conjugate을 구합니다.

In [12]:
# Pauli operator의 complex conjugate 구하기
p = Pauli('ZX')
print(f"Conjugate of {p} is {p.conjugate()}")
print(f"Conjugate of {p2} is {p2.conjugate()}")

Conjugate of ZX is ZX
Conjugate of Y is -Y


#### delete

주어진 Pauli operator의 특정 qubit들을 제거합니다. 결과로 얻어지는 Pauli operator는 더 적은 수의 qubit들을 가지게 됩니다.

In [13]:
# Pauli operator의 특정 qubit들을 제거하기
qubit_index = 0
delete_0 = p.delete(qubit_index)

print(f"{p} after deleting qubit {qubit_index} is {delete_0}")

ZX after deleting qubit 0 is Z


#### insert

주어진 Pauli operator의 특정 qubit들에 다른 Pauli operator들을 더합니다.

In [ ]:
# Pauli operator에 다른 Pauli operator들을 더하기
qubit_index = 1
insert_0 = delete_0.insert(qubit_index, Pauli('X'))

print(f"{delete_0} after inserting X at qubit {qubit_index} is {insert_0}")

Z after inserting X at qubit 1 is XYZ


#### to_instruction

Pauli operator를 quantum circuit에 쓰일수 있는 quantum instruction으로 바꿉니다.

In [15]:
# Pauli operator를 quantum instruction으로 바꾸기
p.to_instruction()

Instruction(name='pauli', num_qubits=2, num_clbits=0, params=['ZX'])

---
## 요약
---

Pauli operator는

1. Pauli stirngs, boolean arrays, quantum circuits, `ScalarOp`으로부터 만들수 있습니다.
2. `to_matrix()`에 의해 matrix로, 또는 `to_label()`에 의해 string으로 바꿀수 있습니다.
3. 제공되는 여러 속성과 함수에 의해 조작될수 있습니다.

---

## 예제

**1) 다음 Qiskit code의 결과는?**

```
import numpy as np
from qiskit.quantum_info import Pauli

z = np.array([True, False, True])   
x = np.array([False, True, True])
p = Pauli((z, x))
print(p)
```

A) ZXY

B) YXZ

C) IY

D) XZY

E) -iYZ


***Answer:***
<Details>
<br/>
B) YXZ

Qubit 0: (Z=True, X=False) → Z operator

Qubit 1: (Z=False, X=True) → X operator

Qubit 2: (Z=True, X=True) → Y operator
</Details>

---

**2. 다음 중 single-qubit Y gate와 같은 결과를 주는 Pauli operator는? (global phase는 다를수 있음.)**

A) ```Pauli('Y')```

B) ```Pauli(([True], [True]))```

C) ```Pauli(([False], [True]))```

D)  ```qc = QuantumCircuit(2)```
       
       qc.y(1)
       
       pauli_quantum_circuit = Pauli(qc)


E) Both A and B

***Answer:***
<Details>
<br/>
E) Both A and B - Pauli('Y') and Pauli(([True], [True])) both create the Y operator, while C creates X  and D Creates YI
</Details>

---

**3) 다음 code의 결과는?**

    from qiskit.quantum_info import Pauli
    
    p = Pauli('ZYX')
    result = p.delete([0,2])
    print(result)

A) XYZ

B) XZ

C) Y

D) ZYX

E) ZX

***Answer:***
<Details>
<br/>
C) Y , qubits 0 and 2 are deleted qubit 1 remains which is Y
</Details>